# Task-specific distillation for BERT
Link:
- https://www.philschmid.de/knowledge-distillation-bert-transformers 

This is an end-to-end task-specific knowledge distillation text-classification example. Here we'll be using Transformers and PyTorch.

Distillation is the process of training a small "student" model to mimic a larger "teacher" model. 

In this example, we will use a [BERT-base](https://huggingface.co/textattack/bert-base-uncased-SST-2) as Teacher (~109 M) and [BERT-Tiny](https://huggingface.co/google/bert_uncased_L-2_H-128_A-2) (4.4 M) as Student. We will use [Text-Classification](https://huggingface.co/tasks/text-classification) as our task-specific knowledge distillation task and the [Stanford Sentiment Treebank v2 (SST-2)](https://huggingface.co/datasets/stanfordnlp/sst2) dataset for training.

They are two different types of knowledge distillation, the Task-agnostic knowledge distillation (right) and the Task-specific knowledge distillation (left). In this example we are going to use the Task-specific knowledge distillation.


In Task-specific knowledge distillation a "second step of distillation" is used to "fine-tune" the model on a given dataset. This idea comes from the [DistilBERT](https://arxiv.org/abs/1910.01108) paper where it was shown that a student performed better than simply finetuning the distilled language model.

## Installation

In [1]:
%pip install transformers==5.20.2 datasets==5.1.0 tensorboard==2.21.0
%pip install accelerate==1.15.0
# run in terminal:
# !sudo apt-get install git-lfs
# or
# brew install git-lfs
# git lfs install 

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Setup and configuration
In this example, we will use BERT-base as Teacher and BERT-Tiny as Student. Our Teacher is already fine-tuned on our dataset, which makes it easy for us to directly start the distillation training job rather than fine-tuning the teacher first to then distill it afterwards.

*IMPORTANT: This example will only work with a Teacher & Student combination where the Tokenizer is creating the same output.*



In [2]:
student_id = "google/bert_uncased_L-2_H-128_A-2"
teacher_id = "textattack/bert-base-uncased-SST-2"

In [3]:
from transformers import AutoTokenizer

/home/imaha/Desktop/study/small-LLM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here we check, that teacher and student divide identical text into the same tokens. Otherwise it won'r work.

In [4]:
# init tokenizer
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_id)
student_tokenizer = AutoTokenizer.from_pretrained(student_id)
 
# sample input
sample = "This is a basic example, with different words to test."
 
# assert results
assert teacher_tokenizer(sample) == student_tokenizer(sample), "Tokenizers haven't created the same output"
 

## Dataset and pre-processing

As Dataset we will use the [Stanford Sentiment Treebank v2 (SST-2)](https://huggingface.co/datasets/stanfordnlp/sst2) a text-classification for `sentiment-analysis`, which is included in the [GLUE benchmark](https://gluebenchmark.com/). The dataset is based on the dataset introduced by Pang and Lee (2005) and consists of 11,855 single sentences extracted from movie reviews. It was parsed with the Stanford parser and includes a total of 215,154 unique phrases from those parse trees, each annotated by 3 human judges. It uses the two-way (positive/negative) class split, with only sentence-level labels.

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer


In [6]:
dataset_id="nyu-mll/glue"
dataset_config="sst2"

In [7]:
dataset = load_dataset(dataset_id,dataset_config)

### Pre-processing & Tokenization

To distill our model we need to convert our "Natural Language" to token IDs.

We are going to use the tokenizer of the Teacher, though we could also go with the Student tokenizer.

In [8]:

tokenizer = AutoTokenizer.from_pretrained(teacher_id)

Additionally we add the `truncation=True` and `max_length=512` to align the length and truncate texts that are bigger than the maximum size allowed by the model.

In [9]:
def process(examples):
    tokenized_inputs = tokenizer(
        examples["sentence"], truncation=True, max_length=512
    )
    return tokenized_inputs

In [10]:
tokenized_datasets = dataset.map(process, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label","labels")
 
tokenized_datasets["test"].features

{'sentence': Value('string'),
 'labels': ClassLabel(names=['negative', 'positive']),
 'idx': Value('int32'),
 'input_ids': List(Value('int32')),
 'token_type_ids': List(Value('int8')),
 'attention_mask': List(Value('int8'))}

### Train split

Ten percent of the SST-2 train split is held out for hyperparameter search and the `alpha=1` comparison. Final models train on the other 90% only. The original validation split is reserved for the reported metrics.

In [11]:
parts = tokenized_datasets["train"].train_test_split(
    test_size=0.1,
    seed=33,
    stratify_by_column="labels",
)
fit_dataset = parts["train"]
selection_dataset = parts["test"]
print(len(fit_dataset), len(selection_dataset), len(tokenized_datasets["validation"]))

60614 6735 872


`input_ids` -- stores tokenized sentences.

## Distilling the model using PyTorch and DistillationTrainer
Now that our `dataset` is processed, we can distill it. Normally, when fine-tuning a transformer model using PyTorch you should go with the `Trainer-API`. The [Trainer](https://huggingface.co/docs/transformers/v4.16.1/en/main_classes/trainer#transformers.Trainer) class provides an API for feature-complete training in PyTorch for most standard use cases.

In our example we cannot use the `Trainer` out-of-the-box, since we need to pass in two models, the `Teacher` and the `Student` and compute the loss for both. But we can subclass the `Trainer` to create a `DistillationTrainer` which will take care of it and only overwrite the [compute_loss](https://github.com/huggingface/transformers/blob/c4ad38e5ac69e6d96116f39df789a2369dd33c21/src/transformers/trainer.py#L1962) method as well as the `init` method. In addition to this we also need to subclass the `TrainingArguments` to include the our distillation hyperparameters.

In [12]:
from transformers import TrainingArguments, Trainer
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

In [13]:
class DistillationTrainingArguments(TrainingArguments):
    def __init__(self, *args, alpha=0.5, temperature=2.0, **kwargs):
        super().__init__(*args, **kwargs)
 
        self.alpha = alpha # to include two losses equally: 
                            # real label of sst and how much 
                            # distributions of student and teacher differ
        self.temperature = temperature # 2.0 by default 
                                        # logits are dividet by this value
                                        # student studies how confident was the teacher
        # because TrainingArguments does not have those args

class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model # adding teacher into the trainer
        # place teacher on same device as student
        self._move_model_to_device(self.teacher,self.model.device) # placing teacher on the same device 
                                                                    # so comparison of logits is valid
        self.teacher.eval() # turn on evaluation regime: no dropout, same logits for the same sentence;
                            # weights are not updated, it also does not get into the optimiser


    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
 
        # compute student output
        outputs_student = model(**inputs)
        student_loss=outputs_student.loss # cross-entropy loss
        # compute teacher output
        with torch.no_grad(): # weights are not being updated
          outputs_teacher = self.teacher(**inputs)
 
        # assert size
        assert outputs_student.logits.size() == outputs_teacher.logits.size()
 
        # Soften probabilities and compute distillation loss
        loss_function = nn.KLDivLoss(reduction="batchmean") 
        loss_logits = (loss_function(
            F.log_softmax(outputs_student.logits / self.args.temperature, dim=-1),
            F.softmax(outputs_teacher.logits / self.args.temperature, dim=-1)) * (self.args.temperature ** 2)) 
                                                                                # multiplication for scaling
        # Return weighted student loss
        loss = self.args.alpha * student_loss + (1. - self.args.alpha) * loss_logits
        return (loss, outputs_student) if return_outputs else loss

Kullback-Leibler loss (`KLDivLoss`):
- how much student's distributtion differs from teacher's
- teacher_p * log(teacher_p / student_p)

`reduction="batchmean"` averages those sums per batch:
$$\frac{1}{m}\sum^m_1(p_T(neg)\log(\frac{p_T(neg)}{p_S(neg)})+p_T(pos)\log(\frac{p_T(pos)}{p_S(pos)}))$$

### Hyperparameter Definition, Model Loading

In [14]:
# create label2id, id2label dicts for nice outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label
 
output_dir = "../models/tiny-bert-sst2"

# define training args
training_args = DistillationTrainingArguments(
    output_dir=output_dir,
    num_train_epochs=7, # n of epoch for whole train-split
    per_device_train_batch_size=128, # how many sentences per step
    per_device_eval_batch_size=128,
    fp16=torch.cuda.is_available(), # mixed precision only on CUDA
    learning_rate=6e-5, # optimiser's step
    seed=33,

    # logging & evaluation strategies
    logging_strategy="epoch", # to get more information to TensorBoard
    eval_strategy="epoch", # accuracy for validation
    save_strategy="epoch", 
    save_total_limit=2, # n of last checkpoints
    load_best_model_at_end=True, 
    metric_for_best_model="accuracy",
    report_to="tensorboard",

    # distilation parameters
    alpha=0.5,
    temperature=4.0
    )
 
# define data_collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # padding fot too short sentences
 
# define model
teacher_model = AutoModelForSequenceClassification.from_pretrained(
    teacher_id,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
 
# define student model
student_model = AutoModelForSequenceClassification.from_pretrained(
    student_id,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

Loading weights: 100%|██████████| 39/39 [00:00<00:00, 5468.27it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing fro

In [15]:
print(training_args.device)

cuda:0


In [16]:
teacher_model.save_pretrained("../models/bert-base-sst2-teacher")
tokenizer.save_pretrained("../models/bert-base-sst2-teacher")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 14.25it/s]


('../models/bert-base-sst2-teacher/tokenizer_config.json',
 '../models/bert-base-sst2-teacher/tokenizer.json')

### Evaluation metric

In [17]:
import numpy as np

In [18]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": (predictions == labels).mean().item()}

### Training

In [19]:
trainer = DistillationTrainer(
    student_model,
    training_args,
    teacher_model=teacher_model,
    train_dataset=fit_dataset,
    eval_dataset=selection_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.509289,0.979581,0.854788
2,0.891100,0.730484,0.884929
3,0.690856,0.640672,0.898144
4,0.587933,0.583625,0.902747
5,0.529607,0.569029,0.908537
6,0.497682,0.547343,0.909725
7,0.480793,0.539894,0.912398


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 167.77it/s]


TrainOutput(global_step=3318, training_loss=0.7410370342123573, metrics={'train_runtime': 284.9268, 'train_samples_per_second': 1489.148, 'train_steps_per_second': 11.645, 'total_flos': 50853794675760.0, 'train_loss': 0.7410370342123573, 'epoch': 7.0})

This first run only checks that distillation trains. It uses 90% of the SST-2 train split and is scored on the held-out 10%. The reported numbers are printed later, on the original validation split: the teacher, the distilled student, and the `alpha=1` control.

### Hyperparameter Search for Distillation parameter alpha & temperature with optuna
The parameter `alpha` & `temparature` in the `DistillationTrainer` can also be used when doing Hyperparamter search to maxizime our "knowledge extraction". As Hyperparamter Optimization framework are we using [Optuna](https://optuna.org/), which has an integration into the Trainer-API. Since the DistillationTrainer is a sublcass of the Trainer we can use the hyperparameter_search without any code changes.

In [21]:
%pip install optuna==5.0.0

Note: you may need to restart the kernel to use updated packages.


To do optimisation we need to define hyperparameter space.

In [22]:
def hp_space(trial):
    return {
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-3 ,log=True),
        "alpha": trial.suggest_float("alpha", 0, 1),
        "temperature": trial.suggest_int("temperature", 2, 30),
        }

In [23]:
def student_init():
    return AutoModelForSequenceClassification.from_pretrained(
        student_id,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )

training_args.output_dir = "../models/tiny-bert-hpo"
training_args.save_strategy = "no"
training_args.load_best_model_at_end = False

trainer = DistillationTrainer(
    model_init=student_init,
    args=training_args,
    teacher_model=teacher_model,
    train_dataset=fit_dataset,
    eval_dataset=selection_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
best_run = trainer.hyperparameter_search(
    n_trials=50,
    direction="maximize",
    hp_space=hp_space
)

print(best_run)

Loading weights: 100%|██████████| 39/39 [00:00<00:00, 10564.32it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing fr

Epoch,Training Loss,Validation Loss,Accuracy
1,1.451094,0.885150,0.871566
2,0.754787,0.639374,0.901262
3,0.568256,0.557317,0.911507
4,0.470680,0.509467,0.920713
5,0.415566,0.492866,0.922049
6,0.377327,0.471935,0.925464
7,0.351283,0.461427,0.925761
8,0.339014,0.453848,0.927691


[I 2026-10-25 11:51:24,246] Trial 0 finished with value: 0.927691165553081 and parameters: {'num_train_epochs': 8, 'learning_rate': 0.00010431534209871124, 'alpha': 0.6054179749517107, 'temperature': 18}. Best is trial 0 with value: 0.927691165553081.
Loading weights: 100%|██████████| 39/39 [00:00<00:00, 14114.92it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING

Epoch,Training Loss,Validation Loss,Accuracy
1,3.160688,2.445102,0.791982
2,2.221177,1.831829,0.831626
3,1.826932,1.601446,0.847958
4,1.638858,1.496871,0.854788
5,1.555802,1.462608,0.855679


[I 2026-10-25 11:55:03,444] Trial 1 finished with value: 0.8556792873051225 and parameters: {'num_train_epochs': 5, 'learning_rate': 2.0984583991949252e-05, 'alpha': 0.3644675284947123, 'temperature': 29}. Best is trial 0 with value: 0.927691165553081.
Loading weights: 100%|██████████| 39/39 [00:00<00:00, 19727.19it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSIN

...

Epoch,Training Loss,Validation Loss,Accuracy
1,1.817797,1.093143,0.898886


[I 2026-10-25 14:28:32,996] Trial 48 pruned. 
Loading weights: 100%|██████████| 39/39 [00:00<00:00, 12175.50it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those pa

Epoch,Training Loss,Validation Loss,Accuracy
1,1.454284,0.848278,0.908686
2,0.638753,0.628756,0.925167
3,0.413683,0.528209,0.936006
4,0.303964,0.506273,0.938085
5,0.236978,0.488759,0.938679
6,0.192900,0.463246,0.941203


[I 2026-10-25 14:33:03,974] Trial 49 pruned. 


BestRun(run_id='19', objective=0.9453600593912398, hyperparameters={'num_train_epochs': 8, 'learning_rate': 0.0008258879417208129, 'alpha': 0.008291728921826391, 'temperature': 16}, run_summary=None)


After finding the best combination we can overwrite hyperparameters .

In [24]:
training_args.save_strategy = "epoch"
training_args.load_best_model_at_end = True

# overwrite initial hyperparameters with from the best_run
for k,v in best_run.hyperparameters.items():
    setattr(training_args, k, v)

# Define a new repository to store our distilled model
best_model_ckpt = "../models/tiny-bert-best"
training_args.output_dir = best_model_ckpt

### Selection scores

Optuna scores each trial on the 10% selection slice. The control uses that best trial's `num_train_epochs`, `learning_rate`, and `temperature`, with `alpha=1` (cross-entropy only, no distillation). Both scores below are last-epoch accuracy on the selection slice, which is the quantity Optuna maximises. The original validation split is not used here.

In [25]:
import copy

selection_distill_accuracy = best_run.objective

control_args = copy.copy(training_args)
control_args.alpha = 1.0
control_args.output_dir = "../models/tiny-bert-alpha1-selection"
control_args.save_strategy = "no"
control_args.load_best_model_at_end = False

control_trainer = DistillationTrainer(
    student_init(),
    control_args,
    teacher_model=teacher_model,
    train_dataset=fit_dataset,
    eval_dataset=selection_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
control_trainer.train()
selection_control_accuracy = control_trainer.evaluate()["eval_accuracy"]

print("selection distill", selection_distill_accuracy)
print("selection alpha=1", selection_control_accuracy)

Loading weights: 100%|██████████| 39/39 [00:00<00:00, 11993.39it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing fr

Epoch,Training Loss,Validation Loss,Accuracy
1,0.338503,0.237072,0.907350
2,0.187342,0.226296,0.919228
3,0.127167,0.236132,0.925167
4,0.092299,0.257531,0.929918
5,0.067980,0.283564,0.928137
6,0.053896,0.288499,0.930215
7,0.041490,0.349274,0.928434
8,0.033495,0.364708,0.927840


Training Loss,Validation Loss,Epoch,Accuracy
0.033495,0.364708,8,0.927840


selection distill 0.9453600593912398
selection alpha=1 0.9278396436525612


### Validation scores

These three numbers are the reported result. Both students train on the other 90% of the train split. The printed student accuracy is the best checkpoint on the original validation split (`metric_for_best_model="accuracy"`), and that checkpoint stays loaded. The teacher is only evaluated.

In [26]:
import copy

final_trainer = DistillationTrainer(
    student_init(),
    training_args,
    teacher_model=teacher_model,
    train_dataset=fit_dataset,
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
final_trainer.train()

def best_accuracy_row(trainer):
    rows = [row for row in trainer.state.log_history if "eval_accuracy" in row]
    return max(rows, key=lambda row: row["eval_accuracy"])

distill_best = best_accuracy_row(final_trainer)
print(
    "distilled validation accuracy",
    distill_best["eval_accuracy"],
    "epoch",
    distill_best["epoch"],
)

final_control_args = copy.copy(training_args)
final_control_args.alpha = 1.0
final_control_args.output_dir = "../models/tiny-bert-alpha1"
final_control = DistillationTrainer(
    student_init(),
    final_control_args,
    teacher_model=teacher_model,
    train_dataset=fit_dataset,
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
final_control.train()
control_best = best_accuracy_row(final_control)
print(
    "alpha=1 validation accuracy",
    control_best["eval_accuracy"],
    "epoch",
    control_best["epoch"],
)

teacher_eval = Trainer(
    teacher_model,
    args=training_args,
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
print("teacher validation", teacher_eval.evaluate())

final_trainer.save_model("../models/tiny-bert-best")
tokenizer.save_pretrained("../models/tiny-bert-best")

# The generated card takes its headline from the last eval row. Append the
# loaded checkpoint so that headline is the best validation accuracy.
final_trainer.state.log_history.append(dict(distill_best))
final_trainer.create_model_card(
    model_name="tiny-bert-sst2",
    finetuned_from="google/bert_uncased_L-2_H-128_A-2",
    dataset="nyu-mll/glue",
    dataset_args="sst2",
    tasks="text-classification",
)

Loading weights: 100%|██████████| 39/39 [00:00<00:00, 8628.43it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing fro

Epoch,Training Loss,Validation Loss,Accuracy
1,1.689037,2.035892,0.809633
2,0.727785,2.292758,0.810780
3,0.477170,2.059277,0.811927
4,0.350515,2.238068,0.813073
5,0.262282,2.330901,0.813073
6,0.207680,2.225502,0.811927
7,0.165461,2.171103,0.816514
8,0.135141,2.134262,0.809633


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 162.97it/s]


distilled validation accuracy 0.8165137614678899 epoch 7.0


Loading weights: 100%|██████████| 39/39 [00:00<00:00, 8257.75it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing fro

...

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 165.57it/s]


alpha=1 validation accuracy 0.805045871559633 epoch 1.0


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.281863,0,0.924312


teacher validation {'eval_loss': 0.28186291456222534, 'eval_accuracy': 0.9243119266055045}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 161.62it/s]
